# Prediccion del Mundial 2026 - Machine Learning + econometria

Notebook auto-actualizable: lee el Excel `Mundial_2026_fuente_datos.xlsx`,
toma los resultados ya cargados como hechos fijos y recalcula las
probabilidades cada vez que se reejecuta (Entorno de ejecucion -> Ejecutar todo).

Flujo profesional de prediccion:

1. Carga y limpieza de datos (48 selecciones, fixture, cuadro final).
2. Construccion de features por partido (rating Elo, ranking y puntos FIFA,
   titulos, apariciones, mejor resultado historico, valor de plantel, edad).
3. Entrenamiento de modelos: Elo probabilistico, Dixon-Coles/Poisson y tres
   modelos ML (logit multinomial, RandomForest, GradientBoosting), mas un
   ensemble.
4. Evaluacion de los modelos por validacion cruzada out-of-fold (log-loss,
   accuracy, Brier) y seleccion automatica del mejor para el pronostico 1/X/2.
5. Simulacion Monte Carlo del torneo (Dixon-Coles como generador de marcadores)
   con desempate oficial FIFA y localia moderada a los anfitriones en
   eliminatorias.

Entrega probabilidades, no recomendaciones de apuestas.

## 1. Preparacion del entorno (clona el repo e instala dependencias en Colab)

In [ ]:
import os, sys

EN_COLAB = 'google.colab' in sys.modules
REPO = 'ML_prediccion_mundial2026'
REPO_URL = 'https://github.com/santiagoriverti/ML_prediccion_mundial2026.git'

# En Colab: clonar el repo (trae src/ y el Excel) e instalar requirements.
if EN_COLAB:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL}
    %cd {REPO}
    !pip install -q -r requirements.txt

# Localizar la carpeta src/ (funciona tanto en Colab como en local).
for cand in ['src', '../src', os.path.join(REPO, 'src')]:
    if os.path.exists(cand):
        sys.path.insert(0, os.path.abspath(cand))
        RAIZ = os.path.dirname(os.path.abspath(cand))
        break
print('Raiz del proyecto:', RAIZ)

## 2. Carga del Excel (siempre la ultima version commiteada)

In [ ]:
import glob, urllib.request
import warnings; warnings.filterwarnings('ignore')

# 1o intenta la raw URL del repo (toma el ultimo commit); 2o archivo local;
# 3o (solo Colab) subida manual.
RAW = ('https://raw.githubusercontent.com/santiagoriverti/'
       'ML_prediccion_mundial2026/main/Mundial_2026_fuente_datos.xlsx')
datos_bytes = None
try:
    datos_bytes = urllib.request.urlopen(RAW, timeout=25).read()
    print('OK - Excel cargado desde la raw URL del repo.')
except Exception as e:
    print('La raw URL fallo (', e, '); busco archivo local .xlsx...')
    locales = glob.glob(os.path.join(RAIZ, '*.xlsx')) + glob.glob('*.xlsx')
    if locales:
        datos_bytes = open(locales[0], 'rb').read()
        print('OK - Excel local:', locales[0])
    elif EN_COLAB:
        from google.colab import files
        subido = files.upload()
        datos_bytes = list(subido.values())[0]
        print('OK - Excel subido manualmente.')
    else:
        raise FileNotFoundError('No se encontro el Excel insumo.')

## 3. Lectura y limpieza de datos

Se cargan las 48 selecciones, el fixture de grupos y el cuadro de eliminatorias.
El loader tolera hojas plantilla vacias, normaliza nombres por `Pais` y descarta
la fila de nota al pie.

In [ ]:
from data_loader import cargar_datos
from features import imputar_rating_base, construir_dataset_partidos

datos = cargar_datos(datos_bytes)
print('Equipos:', datos.meta['n_equipos'],
      '| Partidos de grupo:', datos.meta['n_partidos_grupo'],
      '| Ya jugados:', datos.meta['n_jugados'],
      '| Sin puntos FIFA (imputados):', datos.meta['n_sin_puntos'])

equipos = imputar_rating_base(datos.equipos)
equipos[['pais', 'grupo', 'confederacion', 'rating_base', 'rating_imputado']] \
    .sort_values('rating_base', ascending=False).head(10)

## 4. Actualizacion del Elo con los resultados ya cargados

Antes de simular, el rating de cada seleccion se mueve con los partidos ya
jugados (los hechos fijan el pronostico), con factor de margen de victoria.

In [ ]:
from simulate import actualizar_elo

equipos = actualizar_elo(equipos, datos.fixture, K=32.0)
dataset = construir_dataset_partidos(equipos, datos.fixture)
print('Dataset por partido:', dataset.shape, '| con resultado (jugados):',
      int(dataset['jugado'].sum()))
equipos[['pais', 'rating_base']].sort_values('rating_base', ascending=False).head(8)

## 5. Variables (features) que usa el modelo

Cada partido se describe por la diferencia A-B de los atributos de las dos
selecciones. Estas son las variables que entran a los modelos supervisados:

In [ ]:
from features import COLUMNAS_FEATURES
import pandas as pd

descripcion = {
    'd_rating': 'Diferencia de rating Elo (Puntos FIFA + actualizacion por resultados)',
    'd_ranking': 'Diferencia de ranking FIFA (signo invertido: mejor = menor)',
    'd_puntos': 'Diferencia de Puntos FIFA',
    'd_titulos': 'Diferencia de titulos mundiales',
    'd_apariciones': 'Diferencia de apariciones en Mundiales',
    'd_mejor_result': 'Diferencia de mejor resultado historico (ordinal)',
    'd_valor_plantel': 'Diferencia de valor de plantel (Transfermarkt, decenas de EUR M)',
    'd_edad': 'Diferencia de edad promedio del plantel',
    'anfitrion': 'Ventaja de localia (1 si A es sede, -1 si B, 0 si no)',
    'altitud': 'Altitud de la sede (placeholder; hoy 0)',
}
pd.DataFrame({'feature': COLUMNAS_FEATURES,
              'descripcion': [descripcion.get(c, '') for c in COLUMNAS_FEATURES]})

## 6. Entrenamiento de los modelos

- Dixon-Coles/Poisson con prior basado en el rating Elo (regularizacion fuerte
  por la muestra chica); es el generador de marcadores de la simulacion.
- Modelos ML supervisados: logit multinomial, RandomForest y GradientBoosting,
  con validacion cruzada estratificada y calibracion.

In [ ]:
from models import DixonColes, entrenar_modelos_ml

dc = DixonColes(equipos).entrenar(dataset)
modelos_ml, reporte = entrenar_modelos_ml(dataset)
print('Partidos de entrenamiento:', reporte['n'],
      '| clases (1/X/2):', reporte.get('clases'))
print('Log-loss CV por modelo ML:', {k: round(v, 3) for k, v in reporte['cv'].items()})

## 7. Evaluacion de modelos y seleccion del mejor

Se compara cada modelo 1/X/2 (Elo, Dixon-Coles, logit, RandomForest,
GradientBoosting y el ensemble) por validacion cruzada OUT-OF-FOLD sobre los
partidos ya jugados: en cada fold se reentrenan Dixon-Coles y los modelos ML
solo con el train y se predice el test (sin fuga de informacion). El mejor
modelo (menor log-loss) se usa para el pronostico por partido.

Nota: la simulacion del torneo necesita un generador de marcadores, rol que
cumple Dixon-Coles; la seleccion del mejor modelo aplica al pronostico 1/X/2.

In [ ]:
from models import evaluar_modelos

# devolver_oof=True entrega tambien las predicciones out-of-fold (oof) y los
# resultados reales (y_eval), que se reutilizan para medir la calibracion abajo.
tabla_eval, mejor_modelo, oof, y_eval = evaluar_modelos(dataset, equipos,
                                                        devolver_oof=True)
print('Mejor modelo 1/X/2 (out-of-fold):', mejor_modelo)
tabla_eval

## 7b. Calibracion del pronostico (backtesting)

Mas alla de cual modelo acierta mas, conviene saber si sus probabilidades son
**confiables**: cuando dice "55%", ?ocurre cerca del 55% de las veces? Con las
mismas predicciones out-of-fold (sin fuga) se arma un *reliability diagram* en
formato uno-contra-resto: se agrupan las probabilidades en tramos y se compara
la probabilidad media predicha (`conf`) con la frecuencia observada
(`frec_obs`). El **ECE** (Expected Calibration Error, menor = mejor) resume la
brecha promedio. Puntos sobre la diagonal = el modelo subestima; por debajo =
sobreestima.

In [ ]:
from models import tabla_calibracion

# ECE de cada modelo con prediccion out-of-fold valida (menor = mejor calibrado).
resumen_calib = []
for m, P in oof.items():
    _, ece_m = tabla_calibracion(P, y_eval)
    if ece_m == ece_m:  # descarta NaN (modelos sin oof completo)
        resumen_calib.append({'modelo': m, 'ECE': ece_m})
resumen_calib = pd.DataFrame(resumen_calib).sort_values('ECE').reset_index(drop=True)
print('Calibracion por modelo (ECE, menor = mejor):')
print(resumen_calib.to_string(index=False))

# Tabla de reliability del modelo elegido para el pronostico.
tabla_calib, ece = tabla_calibracion(oof[mejor_modelo], y_eval)
print(f'\nReliability del modelo "{mejor_modelo}"  |  ECE = {ece}')
tabla_calib

## 8. Pronostico por partido pendiente (con el mejor modelo)

Para cada partido aun no jugado: P(1/X/2) del mejor modelo seleccionado, goles
esperados (Dixon-Coles) y marcador mas probable.

In [ ]:
from models import pronostico_partidos

tabla_partidos = pronostico_partidos(dataset, equipos, dc, modelos_ml,
                                     modelo=mejor_modelo)
print('Partidos pendientes de jugar:', len(tabla_partidos),
      '| modelo usado:', mejor_modelo)
tabla_partidos.head(20)

## 9. Simulacion Monte Carlo del torneo

Completa los partidos no jugados, resuelve los grupos con el desempate oficial
FIFA, asigna los 8 mejores terceros, arma el cuadro y simula hasta la final
(prorroga/penales por fuerza). Los anfitriones reciben localia moderada en
eliminatorias. Los partidos ya jugados quedan fijos.

In [ ]:
from simulate import simular_torneo

N_SIMS = 20000
resultados = simular_torneo(equipos, datos.fixture, datos.bracket, dc,
                            n_sims=N_SIMS, semilla=2026)
print('Corridas:', resultados['n_sims'])

## 10. Ranking de probabilidad de ser campeon

In [ ]:
tabla_campeon = resultados['campeon'].copy()
tabla_campeon['prob_campeon_%'] = (tabla_campeon['prob_campeon'] * 100).round(2)
tabla_campeon[['pais', 'prob_campeon_%']].head(20)

## 11. Probabilidad de alcanzar cada ronda

In [ ]:
cols_ronda = ['pais', 'prob_32avos', 'prob_16avos', 'prob_Cuartos',
              'prob_Semifinales', 'prob_Final', 'prob_campeon']
av = resultados['avance'][cols_ronda].copy()
for c in cols_ronda[1:]:
    av[c] = (av[c] * 100).round(1)
av.head(16)

## 12. Cuadro de eliminatorias del escenario mas probable

Escenario unico y consistente (cada seleccion aparece una sola vez): completa
los partidos de grupo pendientes con su marcador esperado y resuelve el cuadro
de 32avos con nombres de seleccion. Es una proyeccion que cambia al recargar
resultados; la simulacion Monte Carlo de arriba mantiene toda la incertidumbre.

In [ ]:
from viz import grafico_campeon, heatmap_avance, grafico_calibracion

ruta1 = grafico_campeon(resultados['campeon'], top=15)
ruta2 = heatmap_avance(resultados['avance'], top=20)
ruta3 = grafico_calibracion(tabla_calib, ece, mejor_modelo)
print('Guardado en:', ruta1, ',', ruta2, 'y', ruta3)

## 13. Graficos (se guardan en `outputs/`)

In [ ]:
from viz import grafico_campeon, heatmap_avance

ruta1 = grafico_campeon(resultados['campeon'], top=15)
ruta2 = heatmap_avance(resultados['avance'], top=20)
print('Guardado en:', ruta1, 'y', ruta2)

## 14. Probabilidades por grupo (gana grupo / clasifica)

In [ ]:
os.makedirs('outputs', exist_ok=True)
tabla_partidos.to_csv('outputs/pronostico_partidos.csv', index=False)
tabla_eval.to_csv('outputs/evaluacion_modelos.csv', index=False)
resumen_calib.to_csv('outputs/calibracion_ece.csv', index=False)
tabla_calib.to_csv('outputs/calibracion_reliability.csv', index=False)
resultados['campeon'].to_csv('outputs/prob_campeon.csv', index=False)
resultados['avance'].to_csv('outputs/prob_avance.csv', index=False)
resultados['grupos'].to_csv('outputs/prob_grupos.csv', index=False)
bracket.to_csv('outputs/bracket_proyectado.csv', index=False)
print('CSVs de salida guardados en outputs/.')

## 15. Guardar resultados a `outputs/`

In [ ]:
os.makedirs('outputs', exist_ok=True)
tabla_partidos.to_csv('outputs/pronostico_partidos.csv', index=False)
tabla_eval.to_csv('outputs/evaluacion_modelos.csv', index=False)
resultados['campeon'].to_csv('outputs/prob_campeon.csv', index=False)
resultados['avance'].to_csv('outputs/prob_avance.csv', index=False)
resultados['grupos'].to_csv('outputs/prob_grupos.csv', index=False)
bracket.to_csv('outputs/bracket_proyectado.csv', index=False)
print('CSVs de salida guardados en outputs/.')

---
## Como actualizar el pronostico

1. Abri `Mundial_2026_fuente_datos.xlsx` y carga los goles de los partidos
   jugados en la hoja Fixture_Grupos (columnas Goles A / Goles B) y/o en
   Eliminatorias.
2. Commitea y pushea el Excel al repo (la raw URL siempre toma el ultimo commit).
3. Reejecuta el notebook (Ejecutar todo). El rating se actualiza con los nuevos
   resultados, se reentrenan y reevaluan los modelos y los partidos pendientes
   se vuelven a simular.